# INST447 — Team 5: The Lobbying Subsidy Tracker
## How Funding Biases Legislators on Key Policy Issues

**Team Members:** Joseph Pierce · Robert Tzou · Kevin Shepherd · Bereket Zeggai  
**Course:** INST447 — Capstone Project  
**Congress:** 119th (2025–2026) | **Election Cycle:** 2024

---

### Research Question
> *"How does funding bias politicians in their decisions for important policies regarding fossil fuels / green energy, defense / Iran war, and data centers / technologies?"*

This notebook answers that question by correlating PAC campaign donations with:
1. Roll-call **voting behavior** on specific bills from the 119th Congress
2. **Legislative sponsorship** patterns by policy area
3. **Network graph** visualization of the donor–legislator relationship

---

## 1. Setup & Imports

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

# Confirm config loaded
from config import TARGET_LEGISLATORS, TARGET_BILLS, ELECTION_CYCLE, CONGRESS_NUMBER

print(f"Election Cycle: {ELECTION_CYCLE}")
print(f"Congress: {CONGRESS_NUMBER}th")
print(f"Target Legislators: {len(TARGET_LEGISLATORS)}")
print(f"Target Bills: {sum(len(v) for v in TARGET_BILLS.values())} across {len(TARGET_BILLS)} topics")

# Quick roster table
pd.DataFrame(TARGET_LEGISLATORS)[['name','party','state','committees']].style.set_caption("Target Legislators")

## 2. Data Collection
### 2a. FEC Donation Data

In [ ]:
from fetch_fec import fetch_all_donations
donations = fetch_all_donations()
print(f"Donation records: {len(donations)}")
donations.head(10)

### 2b. Congress.gov Legislative Activity

In [ ]:
from fetch_congress import fetch_all_legislation
legislation = fetch_all_legislation()
print(f"Legislation records: {len(legislation)}")
legislation.head(10)

### 2c. Roll-Call Vote Records

In [ ]:
from fetch_votes import fetch_all_votes
votes = fetch_all_votes()
print(f"Vote records: {len(votes)}")
votes.head(15)

In [ ]:
# Summary: how each legislator voted on each bill
pivot = votes.pivot_table(
    index=['legislator_name','party','state'],
    columns='bill_name',
    values='vote',
    aggfunc='first'
).reset_index()
pivot

## 3. Data Wrangling

Merge donations, legislation, and votes into one analysis table.

In [ ]:
from merge_data import merge_datasets
merged = merge_datasets()

print(f"Merged records: {len(merged)}")
print(f"Columns: {list(merged.columns)}")
merged.head(15)

In [ ]:
# Schema summary
merged.dtypes.to_frame('dtype').join(merged.describe(include='all').T[['count','mean','max']])

## 4. Exploratory Analysis

Statistical charts: scatter regression, party breakdown, committee effects.

In [ ]:
from visualize import (
    plot_scatter_regression, plot_top_recipients, 
    plot_donations_by_party, plot_committee_effect
)

plot_scatter_regression(merged)
img = mpimg.imread('output/01_scatter_regression.png')
plt.figure(figsize=(18,6)); plt.imshow(img); plt.axis('off')
plt.title('PAC Donations vs. Relevant Bills Sponsored', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
plot_top_recipients(merged)
img = mpimg.imread('output/02_top_recipients.png')
plt.figure(figsize=(12,8)); plt.imshow(img); plt.axis('off')
plt.title('Top Recipients by Industry PAC Donations', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
plot_donations_by_party(merged)
img = mpimg.imread('output/04_donations_by_party.png')
plt.figure(figsize=(18,6)); plt.imshow(img); plt.axis('off')
plt.title('Industry PAC Donations by Party', fontsize=14)
plt.tight_layout(); plt.show()

## 5. Network Graphs

The primary visualization: donor–legislator networks showing funding relationships alongside voting alignment.

In [ ]:
from visualize_network import generate_all_networks
generate_all_networks()

In [ ]:
topics = ['fossil_fuels', 'defense_iran', 'data_centers_tech']
fig, axes = plt.subplots(1, 3, figsize=(30, 12))
for ax, slug in zip(axes, topics):
    path = f'output/network_{slug}.png'
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img); ax.axis('off')
        ax.set_title(slug.replace('_',' ').title(), fontsize=12)
plt.suptitle('Donor–Legislator Network Graphs by Policy Topic', fontsize=16, y=1.02)
plt.tight_layout(); plt.show()

### Network Graph Interpretation

**Node legend:**
- 🔵 Blue nodes = Democratic legislators
- 🔴 Red nodes = Republican legislators  
- ◆ Colored diamonds = PAC/donor nodes (color = industry)
- Edge thickness = donation amount

**Column layout:**
- Left column: legislators who voted **with** industry position
- Right column: legislators who voted **against** industry position
- Center: legislators with no recorded vote on these bills

*Analyze each graph to see whether there is a visible pattern between edge thickness (money received) and node position (voting alignment).*

## 6. Statistical Analysis

In [ ]:
from analysis import run_analysis
report = run_analysis()

In [ ]:
# Correlation summary table
from scipy.stats import pearsonr, spearmanr
rows = []
for industry in merged['industry'].unique():
    sub = merged[merged['industry'] == industry]
    x = sub['industry_donation_total']
    y = sub['total_relevant_bills']
    mask = x.notna() & y.notna()
    if mask.sum() >= 3:
        r_p, p_p = pearsonr(x[mask], y[mask])
        r_s, p_s = spearmanr(x[mask], y[mask])
        rows.append({
            'Industry': industry,
            'Pearson r': round(r_p, 3),
            'Pearson p': round(p_p, 4),
            'Spearman ρ': round(r_s, 3),
            'Spearman p': round(p_s, 4),
            'Significant (p<0.05)': (p_p < 0.05) or (p_s < 0.05),
        })
pd.DataFrame(rows)

## 7. Vote Alignment Analysis

This section directly tests the research question: does more funding correlate with more industry-aligned voting?

In [ ]:
# Merge vote alignment with donation totals
vote_merge = pd.merge(
    merged[['bioguide_id','legislator_name','party','state','industry','industry_donation_total']],
    votes[['bioguide_id','topic','aligned_with_industry']].rename(columns={'topic':'industry'}),
    on=['bioguide_id','industry'], how='inner'
)

vote_merge['aligned_with_industry'] = vote_merge['aligned_with_industry'].astype(float)

# Group stats
summary = vote_merge.groupby(['industry','aligned_with_industry']).agg(
    count=('bioguide_id','count'),
    mean_donation=('industry_donation_total','mean')
).reset_index()
summary['aligned_label'] = summary['aligned_with_industry'].map({1.0:'Aligned (Yea)', 0.0:'Not Aligned (Nay)'})
summary

In [ ]:
# Bar: mean donation for aligned vs not aligned, per industry
import matplotlib.pyplot as plt
import numpy as np

industries = vote_merge['industry'].unique()
fig, axes = plt.subplots(1, len(industries), figsize=(16, 5), sharey=False)
colors = {'Aligned (Yea)':'#00d4aa', 'Not Aligned (Nay)':'#e74c3c'}

for ax, ind in zip(axes, industries):
    sub = summary[summary['industry'] == ind]
    labels = sub['aligned_label'].values
    vals   = sub['mean_donation'].values
    bar_colors = [colors.get(l,'#888') for l in labels]
    ax.bar(labels, vals, color=bar_colors, edgecolor='white', alpha=0.85)
    ax.set_title(ind, fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean PAC Donation ($)')
    ax.tick_params(axis='x', rotation=15)
    ax.set_facecolor('#1a1a2e')
    
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle('Mean Donations: Industry-Aligned vs. Non-Aligned Voters', fontsize=14, color='white')
plt.tight_layout(); plt.show()

print("\nInterpretation: Legislators who voted in the industry's favor received (on average) more in donations?")
for _, row in summary.iterrows():
    print(f"  {row['industry']:25s}  {row['aligned_label']:25s}  ${row['mean_donation']:,.0f}")

## 8. Conclusions

### Research Question Answer

*"How does funding bias politicians in their decisions for important policies regarding fossil fuels / green energy, defense / Iran war, and data centers / technologies?"*

**Findings summary** *(update after running pipeline)*:

| Finding | Evidence |
|---------|----------|
| Overall correlation between PAC donations and bill sponsorship | Pearson r (see Section 6) |
| Vote alignment pattern by industry | Bar chart in Section 7 |
| Committee membership amplifier effect | Committee effect chart (Section 4) |
| Party-level donation bias | Party breakdown chart (Section 4) |
| Network structure (dense vs sparse donors) | Network graphs (Section 5) |

### Limitations
- Sample size: 15 legislators (freshmen only; non-freshmen excluded)
- Bill selection: only 5 target bills identified so far in the 119th Congress
- Some roll-call votes may return "Not Voting" due to API coverage gaps
- Correlation ≠ causation: donations may follow voting alignment rather than causing it

## 9. References

- Goel, P., Resnik, P., & Miler, K. (2023). Donor activity is associated with US legislators' attention to political issues. *PLOS ONE*. https://pmc.ncbi.nlm.nih.gov/articles/PMC10511130/
- OpenFEC API. Federal Election Commission. https://api.open.fec.gov/developers/
- Congress.gov API. U.S. Congress. https://api.congress.gov/
- House Clerk Roll Call Votes. U.S. House of Representatives. https://clerk.house.gov/evs/2025/index.asp